## Tips

- **Start simple** — a single GRU with 64 units is a good first attempt.
  Only add complexity (stacking, dropout, bidirectional) if the simple model
  is clearly underfitting or overfitting.

- **Use many epochs** — RNNs on time series converge slowly. 10 or 20 epochs
  is almost never enough. Use **at least 100 epochs** and let early stopping
  decide when to stop. The best checkpoint is saved automatically by
  `ModelCheckpoint` — you will not miss the optimal point even if you
  train for too long.

- **Watch the train/val gap** — if train MAE is much lower than val MAE
  you are overfitting. Add dropout, reduce `hidden_size`, or increase
  early stopping patience to give regularization more time to work.

- **Use TensorBoard** — run `tensorboard --logdir runs` in your terminal
  to monitor train and val curves in real time. A healthy training curve
  shows both losses decreasing together. A diverging val curve means overfitting.

- **Learning rate matters** — if the loss is not decreasing after the first
  few epochs, try a lower learning rate (`1e-4` instead of `1e-3`).
  Use `ReduceLROnPlateau` to automatically decay the learning rate when
  val MAE stops improving.

- **The sklearn baseline is hard to beat** — `HistGradientBoosting` with
  explicit lag features is a very strong baseline for tabular time series.
  This is not a failure of the RNN — it reflects a fundamental difference
  between the two approaches: tree models get temporal information from
  hand-crafted features, while RNNs must learn it from raw sequences.
  Getting within 10 bikes/hour of the sklearn baseline (< 44 bikes/hour)
  is an excellent result.

In [1]:
import os
import numpy as np
import joblib

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

In [2]:
save_dir = "data/bike_processed"

# --- Load arrays ---
raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
counts = np.load(os.path.join(save_dir, "counts.npy"))
count_stats = np.load(os.path.join(save_dir, "count_stats.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
naive_mae   = np.load(os.path.join(save_dir, "naive_mae.npy"))
preprocessor = joblib.load(os.path.join(save_dir, "preprocessor_rnn.pkl"))

# --- Recover split sizes ---
num_train = len(train_idx)
num_val   = len(val_idx)
num_test  = len(test_idx)

# --- Count stats in the training set ---
count_mean  = count_stats[0]
count_std   = count_stats[1]

# --- Sanity check ---
print(f"raw_data shape:    {raw_data.shape}")
print(f"counts shape: {counts.shape}")
print(f"train/val/test:    {num_train} / {num_val} / {num_test}")
print(f"counts range: {counts.min():.0f} to {counts.max():.0f} bikes/hour")
print(f"\nNaive baseline — Val MAE: {naive_mae[0]:.2f} | Test MAE: {naive_mae[1]:.2f} bikes/hour")

raw_data shape:    (17210, 28)
counts shape: (17210,)
train/val/test:    12047 / 2581 / 2582
counts range: 1 to 977 bikes/hour

Naive baseline — Val MAE: 93.26 | Test MAE: 80.78 bikes/hour


In [3]:
count_norm = (counts - count_mean) / count_std

In [4]:
class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [5]:
sampling_rate   = 1    # already hourly
sequence_length = 24   # look back 24 hours
delay           = 1    # predict 1 hour ahead
batch_size      = 256

train_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train,
)
val_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train, end_index=num_train + num_val,
)
max_start = len(raw_data) - max(
    (sequence_length - 1) * sampling_rate + 1,
    delay + 1,
)
test_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train + num_val, end_index=max_start,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# --- Sanity check ---
inputs, targets = next(iter(train_loader))
print(f"Input shape:  {inputs.shape}")   # (256, 24, num_features)
print(f"Target shape: {targets.shape}")  # (256,)
print(f"Target range: {targets.min():.0f} to {targets.max():.0f}")

Input shape:  torch.Size([256, 24, 28])
Target shape: torch.Size([256])
Target range: -1 to 3


In [6]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    total_mae = 0.0
    n = 0
    with torch.set_grad_enabled(training):
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds = model(inputs)
            loss = criterion(preds, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            total_mae += torch.sum(torch.abs(preds - targets)).item()
            n += inputs.size(0)
    mae_bikes = (total_mae / n) * count_std
    return total_loss / n, mae_bikes


class ModelCheckpoint:
    def __init__(self, filepath, monitor="val_mae", mode="min", verbose=True):
        self.filepath = filepath
        self.monitor = monitor
        self.verbose = verbose
        self.best = float("inf") if mode == "min" else float("-inf")
        self.mode = mode

    def step(self, metrics, model=None):
        value = metrics[self.monitor]
        improved = value < self.best if self.mode == "min" else value > self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  Best model saved ({self.monitor}: {value:.2f} bikes/hour)")
        return improved


class EarlyStopping:
    def __init__(self, monitor="val_mae", patience=10, min_delta=0.5, mode="min"):
        self.monitor = monitor
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = float("inf") if mode == "min" else float("-inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value = metrics[self.monitor]
        improved = value < self.best - self.min_delta if self.mode == "min" else value > self.best + self.min_delta
        if improved:
            self.best = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  Early stopping (no improvement for {self.patience} epochs)")
        return improved


class ReduceLROnPlateauCB:
    def __init__(self, optimizer, monitor="val_mae", patience=3, factor=0.5):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=patience, factor=factor
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])

In [7]:
class GRUModel(nn.Module):
    def __init__(self, num_features, hidden_size=64, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=num_features,
            hidden_size=hidden_size,
            batch_first=True,
            dropout=dropout if dropout else 0,
        )
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.linear(out[:, -1, :]).squeeze(-1)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = GRUModel(num_features=raw_data.shape[-1], hidden_size=64, dropout=0.2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
writer = SummaryWriter(log_dir="runs/bike_gru")

checkpoint_path = "bike_gru_best.pt"
callbacks = [
    ModelCheckpoint(checkpoint_path, monitor="val_mae"),
    EarlyStopping(monitor="val_mae", patience=10, min_delta=0.5),
    ReduceLROnPlateauCB(optimizer, monitor="val_mae", patience=3, factor=0.5),
]

max_epochs = 100
for epoch in range(1, max_epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_mae = run_epoch(model, val_loader, criterion)

    metrics = {
        "train_loss": train_loss,
        "train_mae": train_mae,
        "val_loss": val_loss,
        "val_mae": val_mae,
    }

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE_bikes_per_hour", {"train": train_mae, "val": val_mae}, epoch)

    print(
        f"Epoch {epoch:03d} | train loss {train_loss:.4f}, MAE {train_mae:.2f} | "
        f"val loss {val_loss:.4f}, MAE {val_mae:.2f} bikes/hour"
    )

    for cb in callbacks:
        if isinstance(cb, ModelCheckpoint):
            cb.step(metrics, model)
        else:
            cb.step(metrics)

    if callbacks[1].should_stop:
        break

writer.close()
print("Training finished.")

Using device: cpu


/Users/darestrepo/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(


Epoch 001 | train loss 0.6100, MAE 88.40 | val loss 1.1534, MAE 129.37 bikes/hour
  Best model saved (val_mae: 129.37 bikes/hour)


Epoch 002 | train loss 0.3866, MAE 66.70 | val loss 0.9804, MAE 114.85 bikes/hour
  Best model saved (val_mae: 114.85 bikes/hour)


Epoch 003 | train loss 0.3508, MAE 64.12 | val loss 0.9078, MAE 106.85 bikes/hour
  Best model saved (val_mae: 106.85 bikes/hour)


...

Epoch 044 | train loss 0.0330, MAE 18.58 | val loss 0.1401, MAE 39.82 bikes/hour


Epoch 045 | train loss 0.0327, MAE 18.51 | val loss 0.1400, MAE 39.07 bikes/hour


Epoch 046 | train loss 0.0324, MAE 18.39 | val loss 0.1434, MAE 40.25 bikes/hour


Epoch 047 | train loss 0.0319, MAE 18.24 | val loss 0.1477, MAE 40.29 bikes/hour


Epoch 048 | train loss 0.0318, MAE 18.21 | val loss 0.1410, MAE 39.55 bikes/hour
  Early stopping (no improvement for 10 epochs)
Training finished.


In [9]:
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
_, val_mae = run_epoch(model, val_loader, criterion)
_, test_mae = run_epoch(model, test_loader, criterion)

print(f"\nGRU — Val MAE: {val_mae:.2f} | Test MAE: {test_mae:.2f} bikes/hour")
print(f"Naive baseline — Val MAE: {naive_mae[0]:.2f} | Test MAE: {naive_mae[1]:.2f} bikes/hour")


GRU — Val MAE: 37.80 | Test MAE: 39.73 bikes/hour
Naive baseline — Val MAE: 93.26 | Test MAE: 80.78 bikes/hour
